# Neural Network Model Testing

# 1. Importing Libraries and the Data

In [188]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow import keras
from collect_log_data import read_log_data

In [189]:
inputs, targets = read_log_data(verbose = True)

Reading prepared data from csv files...
Prepared data collected


# 2. Testing a Basic Neural Network

Below is some code involving a very basic version of the structure this neural network may end up having. It's purpose is to serve as a sanity check to make sure everything is working okay and to look for any problems that might arise.

'ReLU' is used as a hidden layer activation function since it is often one of the best performing hidden layer activation functions. 'Softmax' is used as the output activation function since it is often the best performing activation function for multiclass-classification, and this neural network design is supposed to output the probabilities of multiple classes. 'Categorical Cross-Entropy' is used as the loss function for the model because it also typically causes models doing multiclass classification to perform the best. Adam is used as the optimizer because of its (relatively) context aware dynamic learning rate, which makes it one of the best performing and most popular optimizers. 15 neurons were chosen for the hidden layer partially arbitrarily, but also because it's halfway between the number of inputs (18) and the number of outputs (12).

In [190]:
hidden_activation_function = 'relu'
output_activation_function = 'softmax'
loss_function = 'categorical_crossentropy'

In [191]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [157]:
train_inputs, test_inputs, train_targets, test_targets = train_test_split(inputs, targets, test_size = 0.1)

In [158]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

In [159]:
def match_predictor_model_evaluation(test_inputs, test_targets):
	test_predictions = nn.predict(test_inputs)

	test_predictions_split = np.hsplit(test_predictions, 2)
	red_score_predictions = np.argmax(test_predictions_split[0], axis = -1)
	blu_score_predictions = np.argmax(test_predictions_split[1], axis = -1)

	expected_scores = np.hsplit(test_targets, 2)
	red_score_targets = np.argmax(expected_scores[0], axis = -1)
	blu_score_targets = np.argmax(expected_scores[1], axis = -1)

	# Original, previous evaluation
	exact_score_accuracy = sum(np.hstack((red_score_predictions, blu_score_predictions)) ==\
									np.hstack((red_score_targets, blu_score_targets))) / test_targets.shape[0] * 100

	mae = mean_absolute_error(np.hstack((red_score_targets, blu_score_targets)),\
								np.hstack((red_score_predictions, red_score_predictions)))

	correct_winners = 0
	for i in range(test_targets.shape[0]):

		if red_score_predictions[i] > blu_score_predictions[i]:
			predicted_winner = 0
		elif red_score_predictions[i] < blu_score_predictions[i]:
			predicted_winner = 1
		else:
			predicted_winner = 2 # tie
		
		if red_score_targets[i] > blu_score_targets[i]:
			actual_winner = 0
		elif red_score_targets[i] < blu_score_targets[i]:
			actual_winner = 1
		else:
			actual_winner = 2 # tie
		
		if predicted_winner == actual_winner:
			correct_winners += 1

	correct_winner_accuracy = correct_winners / test_targets.shape[0] * 100
	return exact_score_accuracy, mae, correct_winner_accuracy

In [160]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 726us/step
Exact Score Accuracy: 38.24%
MAE: 1.43
Correct Winner Accuracy: 52.46%


The model successfully trained and predicted and seems to get a decent MAE and accuracy when guessing the correct winning team of a match. It's exact score accuracy is approximately the same as the previous design of this neural network. This accuracy is debatably acceptable since it's for whether or not it guessed the exact correct scores of both teams for a match, and 38% is arguably decently accurate for that. Especially when the MAE suggests that it's only about 1 point and a half off on average.

# 3. Testing Number of Epochs

This test is to see if increasing the number of epochs besides 100 will make any difference on the performance of the basic test model above. This will give an idea of how many epochs it will be good to use for hyperparameter testing with neuron counts and layer counts.

In [161]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [162]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 1000, verbose = False)

In [163]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step
Exact Score Accuracy: 33.11%
MAE: 1.43
Correct Winner Accuracy: 52.25%


Since doing 1,000 epochs instead of 100 yielded approximately the same results, that means it's likely not necessary to do more than 100 epochs for testing.

# 4. Testing Layer Counts and Hidden Neuron Counts

Below is code to iterate through various numbers of hidden layers and neurons in each hidden layer to test each of them in order to find the best combination.

In [182]:
def print_model_details(model_details):
	print("Model:")
	print(f"Hidden Layers: {model_details[0]}, Hidden Neuron Counts: {model_details[1]}")

def print_bar():
	print("----------")

min_neurons = 10
max_neurons = 20

models = [
	(1, [10]),
	(1, [15]),
	(1, [20]),
	(2, [10, 10]),
	(2, [15, 10]),
	(2, [20, 10]),
	(2, [10, 15]),
	(2, [15, 15]),
	(2, [20, 15]),
	(2, [10, 20]),
	(2, [15, 20]),
	(2, [20, 20]),
	(3, [10, 10, 10]),
	(3, [15, 10, 10]),
	(3, [20, 10, 10]),
	(3, [10, 15, 10]),
	(3, [15, 15, 10]),
	(3, [20, 15, 10]),
	(3, [10, 20, 10]),
	(3, [15, 20, 10]),
	(3, [20, 20, 10]),
	(3, [10, 10, 15]),
	(3, [15, 10, 15]),
	(3, [20, 10, 15]),
	(3, [10, 15, 15]),
	(3, [15, 15, 15]),
	(3, [20, 15, 15]),
	(3, [10, 20, 15]),
	(3, [15, 20, 15]),
	(3, [20, 20, 15]),
	(3, [10, 10, 20]),
	(3, [15, 10, 20]),
	(3, [20, 10, 20]),
	(3, [10, 15, 20]),
	(3, [15, 15, 20]),
	(3, [20, 15, 20]),
	(3, [10, 20, 20]),
	(3, [15, 20, 20]),
	(3, [20, 20, 20]),
]

In [192]:
best_exact_score = 0
best_mae = float('inf')
best_winner_accuracy = 0

best_exact_score_model = None
best_mae_model = None
best_winner_accuracy_model = None

for model in models:
	nn = keras.models.Sequential(name = "Basic_Neural_Network")

	nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
	for i in range(model[0]):
		nn.add(keras.layers.Dense(units = model[1][i], activation = hidden_activation_function, name = f"Hidden_Layer_{i+1}"))
	nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
	optimizer = keras.optimizers.Adam(learning_rate = 0.00001)

	nn.compile(optimizer = optimizer, loss = loss_function)

	nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

	exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)
	print(f"Hidden Neuron Counts: {model[1]} (Hidden Layers: {model[0]})")
	print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
	print(f"MAE: {mae:.2f}")
	print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

	if exact_score_accuracy > best_exact_score:
		best_exact_score = exact_score_accuracy
		best_exact_score_model = (model[0], model[1].copy())
		print_bar()
		print(f"New best Exact Score: {best_exact_score}")
		print_model_details(best_exact_score_model)

	if mae < best_mae:
		best_mae = mae
		best_mae_model = (model[0], model[1].copy())
		print_bar()
		print(f"New best MAE: {best_mae}")
		print_model_details(best_mae_model)

	if correct_winner_accuracy > best_winner_accuracy:
		best_winner_accuracy = correct_winner_accuracy
		best_winner_accuracy_model = (model[0], model[1].copy())
		print_bar()
		print(f"New best Winner Accuracy: {best_winner_accuracy}")
		print_model_details(best_winner_accuracy_model)

print(f"Best Exact Score: {best_exact_score}")
print_model_details(best_exact_score_model)

print(f"Best MAE: {best_mae}")
print_model_details(best_mae_model)

print(f"Best Winner Accuracy: {best_winner_accuracy}")
print_model_details(best_winner_accuracy_model)

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step
Hidden Neuron Counts: [10] (Hidden Layers: 1)
Exact Score Accuracy: 39.16%
MAE: 1.72
Correct Winner Accuracy: 37.36%
----------
New best Exact Score: 39.15763135946622
Model:
Hidden Layers: 1, Hidden Neuron Counts: [10]
----------
New best MAE: 1.719557964970809
Model:
Hidden Layers: 1, Hidden Neuron Counts: [10]
----------
New best Winner Accuracy: 37.36447039199333
Model:
Hidden Layers: 1, Hidden Neuron Counts: [10]
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 738us/step
Hidden Neuron Counts: [15] (Hidden Layers: 1)
Exact Score Accuracy: 38.91%
MAE: 1.77
Correct Winner Accuracy: 44.91%
----------
New best Winner Accuracy: 44.91242702251876
Model:
Hidden Layers: 1, Hidden Neuron Counts: [15]
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 708us/step
Hidden Neuron Counts: [20] (Hidden Layers: 1)
Exact Score Accuracy: 36.91%
MAE: 1.91
Correct Winner Accuracy: 49.96%
----------
New best Winner Accuracy: 49.9582985821518
Model:
Hidden Layers: 1, Hidden Neuron Counts: [20]
75/75 ━━